# Analysis of random corrupted ICL demonstrations

This notebook reproduces the results of section 4.1 of the paper.
Before running this notebook you need:
1. Results of in_context NED for all dataset, random corruption schemes
2. Results of zero_shot and 10_shot NED for all datasets.

For more informatin on on how to run these experiments, please refer to the README file.

In [1]:
import json
import os

import pandas as pd
from utils.config import load_experiment_result_config
from utils.plotting import bar_plot

In [17]:
seed = 12345
results_folder = "outputs"
dataset_list = ["chemprotgene", "chemprotchem", "bc5chem", "bc5disease", "bc2gm"]
random_corruption_schemes = [
    "random_id_labels",
    "swapped_id_labels",
    "random_ood_labels",
    "random_ood_labels_from_text",
    "corrupted_ood_text",
    "corrupted_and_shuffled_ood_text",
    "corrupted_ood_text_and_labels",
    "corrupted_and_shuffled_ood_text_and_labels",
]
analysis_demo_retrieval = "knn"
baseline_demo_retrieval = "knn"

In [18]:
setups = []
for dataset in dataset_list:
    for corruption_type in random_corruption_schemes:
        setups.append(
            {
                "dataset": dataset,
                "corruption": corruption_type,
                "num_shots": 10,
                "demo_retrieval": analysis_demo_retrieval,
                "experiment_name": corruption_type.replace("-", " ")
                .replace("id", "ID")
                .replace("ood", "OOD"),
                "results_path": "",
            }
        )

    setups.extend(
        [
            {
                "dataset": dataset,
                "corruption": None,
                "num_shots": 10,
                "demo_retrieval": baseline_demo_retrieval,
                "experiment_name": "Gold Label",
                "results_path": "",
            },
            {
                "dataset": dataset,
                "corruption": None,
                "num_shots": 0,
                "demo_retrieval": None,
                "experiment_name": "No Demo",
                "results_path": "",
            },
        ]
    )

sorted_experiments_desc = sorted(os.listdir(results_folder), reverse=True)
for day in sorted_experiments_desc:
    for time in sorted(os.listdir(os.path.join(results_folder, day)), reverse=True):
        hydra_dict = load_experiment_result_config(
            os.path.join(results_folder, day, time, ".hydra"), "hydra"
        )
        runtime_cfg_path = hydra_dict["hydra"]["runtime"]["config_sources"][1][
            "path"
        ].split("/")[-1]

        if runtime_cfg_path == "analysis_random_corrupted_demos":
            cfg_dict = load_experiment_result_config(
                os.path.join(results_folder, day, time, ".hydra"), "config"
            )
            dataset = cfg_dict["data"]["dataset"]
            corruption = cfg_dict.get("corruption_type", {}).get("name", None)
            num_shots = cfg_dict.get("demonstration_retrieval", {}).get(
                "num_shots", None
            )
            demo_retrieval = cfg_dict["demonstration_retrieval"].get("method", None)
            if dataset in dataset_list and corruption in random_corruption_schemes:
                for setup in setups:
                    if (
                        setup["dataset"] == dataset
                        and setup["corruption"] == corruption
                        and setup["num_shots"] == num_shots
                        and setup["demo_retrieval"] == demo_retrieval
                        and setup["results_path"] == ""
                    ):
                        setup["results_path"] = os.path.join(
                            results_folder, day, time, "ner_iob_evaluation.json"
                        )
                        break

for setup in setups:
    if setup["results_path"] == "":
        print(f"Missing results for {setup['dataset']} {setup['corruption']}")
        continue

In [19]:
results_dict = {
    "dataset": [],
    "experiment_name": [],
    "Precision": [],
    "Recall": [],
    "Micro F1": [],
}
for setup in setups:
    results = json.load(open(setup["results_path"], "r"))
    results_dict["dataset"].append(setup["dataset"])
    results_dict["experiment_name"].append(setup["experiment_name"])
    results_dict["Precision"].append(results["micro avg"]["precision"])
    results_dict["Recall"].append(results["micro avg"]["recall"])
    results_dict["Micro F1"].append(results["micro avg"]["f1-score"])

results_df = pd.DataFrame(results_dict)

In [ ]:
bar_plot(results_df, "mistral", "Micro F1")